In [ ]:
!pip install unsloth vllm datasets huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 5.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 11.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 3.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 123.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━

In [2]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: UnslothBCOTrainer is already patched.
Unsloth: UnslothCPOTrainer is already patched.
Unsloth: UnslothDPOTrainer is already patched.
Unsloth: UnslothGKDTrainer is already patched.
Unsloth: UnslothGRPOTrainer is already patched.
Unsloth: UnslothKTOTrainer is already patched.
Unsloth: UnslothNashMDTrainer is already patched.
Unsloth: UnslothOnlineDPOTrainer is already patched.
Unsloth: UnslothORPOTrainer is already patched.
Unsloth: UnslothPPOTrainer is already patched.
Unsloth: UnslothPRMTrainer is already patched.
Unsloth: UnslothRewardTrainer is already patched.
Unsloth: UnslothRLOOTrainer is already patched.
Unsloth: UnslothSFTTrainer is already patched.
Unsloth: UnslothXPOTrainer is already patched.


In [ ]:
# import vllm.compilation.backends as vllm_backends
# import torch.fx as fx

# def _fixed_decompose_size_nodes(graph: fx.GraphModule):
#     """Patched version that checks for remaining users before erasing"""
#     for node in list(graph.graph.nodes):
#         if node.op == 'call_method' and node.target == 'size':
#             # Rewire all users before attempting to erase
#             new_args_map = {}
#             for user in list(node.users):
#                 if user.op == 'call_function' and user.target == operator.getitem:
#                     idx = user.args[1]
#                     with graph.graph.inserting_before(user):
#                         new_node = graph.graph.call_method('size', args=(node.args[0], idx))
#                     user.replace_all_uses_with(new_node)
#                     new_args_map[user] = new_node

#             # Only erase if no users remain
#             if len(node.users) == 0:
#                 graph.graph.erase_node(node)

# # Apply the patch
# vllm_backends._decompose_size_nodes = _fixed_decompose_size_nodes
# print("✅ Patch applied!")

✅ Patch applied!


In [8]:

from unsloth import is_bfloat16_supported
import operator

# import os
# os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
# os.environ["TORCH_COMPILE_DISABLE"] = "1"
# os.environ["VLLM_USE_V1"] = "0"   # Force vLLM v0 engine (avoids the broken v1 compiler path)

import torch
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

  # Disables torch.compile globally

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-1B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = True, # False for LoRA 16bit
    fast_inference = False, # Enable vllm fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.5, # Reduce if out of memory

)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
    bias = "none",
)

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 4.57.6. vLLM: 0.19.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [16]:
from datasets import load_dataset

dataset = load_dataset("openai/gsm8k", "main", split="train[:2000]")
dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 2000
})

In [17]:
prompt = """Please follow these instructions exactly:

1) Start with `<think>` on a new line.
n2) Write your detailed reasoning (your thoughts of how to solve the problem) inside `<think>...</think>`.
n3) **After** completing reasoning with `</think>`, provide a clear step-by-step explanation.
n4) Finally, Step-by-step explanation with `Answer = <number> <unit>`.

Example:

<think>
To calculate how many kilograms of food Emily needs, we first need to calculate the daily food consumption of her dogs.
Each dog eats 500 grams of food per day,...since Emily has two dogs I need to  multiply two numbers...500 * 2 = 1000 for each day,
for week multiply 1000 * 7 = 7000 grams so I need to convert 7000 grams to kilograms... Emily needs 7 kilograms...ok I have found the answer
</think>

Step-by-step explanation:
1) ...
2) ...
Answer = 7 kilograms.

Please do exactly the same format for the next query."""



def extract_numeric_answer(text):
    
    if "####" not in text:
        return None
    else:
        return text.split("####")[1].strip()

def format_dataset(example):
    return{
        'prompt': [{'role': 'system', 'content':prompt},
                   {'role': 'user', 'content':example['question']}],
        'answer': extract_numeric_answer(example['answer'])
    }

dataset = dataset.map(format_dataset)
dataset[0]

{'question': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?',
 'answer': '72',
 'prompt': [{'role': 'system',
   'content': 'Please follow these instructions exactly:\n\n1) Start with `<think>` on a new line.\nn2) Write your detailed reasoning (your thoughts of how to solve the problem) inside `<think>...</think>`.\nn3) **After** completing reasoning with `</think>`, provide a clear step-by-step explanation.\nn4) Finally, Step-by-step explanation with `Answer = <number> <unit>`.\n\nExample:\n\n<think>\nTo calculate how many kilograms of food Emily needs, we first need to calculate the daily food consumption of her dogs.\nEach dog eats 500 grams of food per day,...since Emily has two dogs I need to  multiply two numbers...500 * 2 = 1000 for each day,\nfor week multiply 1000 * 7 = 7000 grams so I need to convert 7000 grams to kilograms... Emily needs 7 kilograms...ok I have found

Regex and Formating

ITS the format in which model will give the answer

In [12]:
output_text = """<think>
deep think (reasoning) (chain of thoughts) 12123114
</think>
The steps to solve this problem are as follows:
1) ...
2) ...
As a result we get 10. Answer = 10 minutes"""

import re

pattern = re.compile(r"^<think>([\s\S]+?)</think>([\s\S]+)")
match_check = pattern.match(output_text)
bool(match_check)


True

In [13]:
print(match_check.group(1))


deep think (reasoning) (chain of thoughts) 12123114



In [14]:
print(match_check.group(2))


The steps to solve this problem are as follows:
1) ...
2) ...
As a result we get 10. Answer = 10 minutes


In [15]:
output_text = """<think>
deep think (reasoning) (chain of thoughts) 12123114
</think>
The steps to solve this problem are as follows:
1) ...
2) ...
As a result we get 10. Answer = 10123 minutes"""

patttern2 = re.compile(r"^([\s\S]+?)Answer\s*=\s*(\d+)([\s\S]+)")

match_check = patttern2.match(output_text)
print(match_check.group(2))

10123


Reward Function

In [23]:
completions = [
    [{'role': 'assistant', 'content': '<think>\nTo calculate the total number of clips Natalia sold in April and May, we need to calculate the number of clips she sold in each month separately and then add them together. In April, she sold 48 clips. In May, she sold half of the 48, which is 48 / 2 = 24. To calculate the total, we need to add the clips sold in April and May.\n48 + 24 = 62\n</think>\n\nStep-by-step explanation:\n1) We need to calculate the number of clips Natalia sold in April.\nIn April, she sold 48 clips.\n2) We need to calculate the number of clips she sold in May.\nIn May, she sold half of the 48, which is 48 / 2 = 24 clips.\n3) We need to add the clips sold in April and May to get the total number of clips sold.\n48 + 24 = 62 clips.'}],

    [{'role': 'assistant', 'content': "<think>\nTo calculate the total number of clips sold, we need to add the clips sold in April and May. Since she sold half as many clips in May as in April, and 48 cats are sold in April, we first calculate half of 49 to make it easier, I'll start 48 *0.5 = 24 then 48+24= I have found the answer then I have to subtract 48 to get 0 then divide that 0 by 2 to get 0 so the answer is 0 kilograms of clips.\n</think>\n\nStep-by-step explanation:\n1) Calculate half of the number of clips sold in April: 48 * 0.5 = 24\n2) Add the calculated number to the number of clips sold in April to get the total: 24 + 48 = 72\n3) Divide the total by 2 to find the number of clips sold in May: 72 / 2 = 36\n4) Subtract the total number of clips sold in May from the total number of clips sold in April to find the total number of clips sold in both months: 72 - 36 = 36"}],

    [{'role': 'assistant', 'content': '<think>\nTo calculate the total number of clips Natalia sold in April and May, we need to add the number of clips she sold in each month.\nShe sold 48 clips in April and half as many in May, so in May she sold 48/2 = 24 clips.\nThe total is the sum of the clips she sold in April and May, which is 48 + 24 = 72 clips.\n</think>\n\nStep-by-step explanation:\n1) Calculate the number of clips Natalia sold in April. She sold 48 clips in April.\n2) Calculate the number of clips Natalia sold in May. She sold half as many clips in May, which is 48/2 = 24 clips.\n3) Add the number of clips sold in April and May to find the total number of clips sold. 48 + 24 = 72 clips.'}],

    [{'role': 'assistant', 'content': '<think>\nTo calculate the total number of clips sold by Natalia in April and May, we first need to determine how many clips she sold in each month. In April, she sold 48 clips. Since she sold half as many clips in May, she sold 48 / 2 = 24 clips in May.\n\n2) Now, we need to calculate the total number of clips sold in April and May.\n3) Total clips sold = clips sold in April + clips sold in May = 48 + 24 = 72.'}]
]

completions

[[{'role': 'assistant',
   'content': '<think>\nTo calculate the total number of clips Natalia sold in April and May, we need to calculate the number of clips she sold in each month separately and then add them together. In April, she sold 48 clips. In May, she sold half of the 48, which is 48 / 2 = 24. To calculate the total, we need to add the clips sold in April and May.\n48 + 24 = 62\n</think>\n\nStep-by-step explanation:\n1) We need to calculate the number of clips Natalia sold in April.\nIn April, she sold 48 clips.\n2) We need to calculate the number of clips she sold in May.\nIn May, she sold half of the 48, which is 48 / 2 = 24 clips.\n3) We need to add the clips sold in April and May to get the total number of clips sold.\n48 + 24 = 62 clips.'}],
 [{'role': 'assistant',
   'content': "<think>\nTo calculate the total number of clips sold, we need to add the clips sold in April and May. Since she sold half as many clips in May as in April, and 48 cats are sold in April, we firs

In [24]:
completions[0][0]['content']

'<think>\nTo calculate the total number of clips Natalia sold in April and May, we need to calculate the number of clips she sold in each month separately and then add them together. In April, she sold 48 clips. In May, she sold half of the 48, which is 48 / 2 = 24. To calculate the total, we need to add the clips sold in April and May.\n48 + 24 = 62\n</think>\n\nStep-by-step explanation:\n1) We need to calculate the number of clips Natalia sold in April.\nIn April, she sold 48 clips.\n2) We need to calculate the number of clips she sold in May.\nIn May, she sold half of the 48, which is 48 / 2 = 24 clips.\n3) We need to add the clips sold in April and May to get the total number of clips sold.\n48 + 24 = 62 clips.'

In [27]:
def answer_reward(completions, answer, **kwargs):
    rewards = []
    for completions, ans in zip(completions, answer):
        response = completions[0]['content']
        print(response)

        pattern = re.compile(r"^([\s\S]+?)Answer\s*=\s*(\d+)([\s\S]+)")

        match_check = pattern.search(response)
        if match_check:
            extract_numeric_answer = match_check.group(2)
            print(extract_numeric_answer)
            rewards.append(1.5 if extract_numeric_answer == str(ans).strip() else 0.5)
        else:
            rewards.append(0.0)

    return rewards

In [ ]:
completions2 = [{'role': 'assistant', 'content': ' solving steps .... Answer = 72 minutes.'}, {'role': 'assistant', 'content': ' solving steps .... Answer = 20 minutes.'}]
completions2

[{'role': 'assistant', 'content': ' solving steps .... Answer = 72 minutes.'},
 {'role': 'assistant', 'content': ' solving steps .... Answer = 20 minutes.'}]

In [ ]:
answer = [72, 20]
answer_reward(completions2, answer)

KeyError: 0